In [ ]:
library(tidyverse)

source("../../evaluation_utils/plots_eda.R")

# Load data

In [ ]:
norm_vs_HGSC = c("GSE6008", "GSE26712", "GSE40595", "GSE69428", "GSE38666", "GSE10971")


Preprocess the data by loading the data from the CSV files and displaying the first few rows of the data.

Save the data plots to the `plots` directory for each sub-datasets.

In [ ]:
all_expression <- NULL
all_metadata <- NULL

i <- 1
for(dataset in norm_vs_HGSC){
    metadata <- read.table(paste0("before/", dataset, "/intermediate/", i, "_metadata_short.tsv"), header = TRUE, sep = "\t")
    expr_data <- read.table(paste0("before/", dataset, "/intermediate/expr_GBacc_MaxVar.tsv"), header = TRUE, sep = "\t") 
    print(paste0("Processing dataset: ", dataset, "platform: ", metadata$platform_id[1]))
    metadata <- metadata %>% mutate(Dataset = dataset) %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage2, Grade2, ClinicopathologicSubtypes, Dataset) %>%
        filter(Status == "normal" | Status == "ovarian tumour" & HistSubtypes == "high-grade serous carcinoma") %>%
        # outliers
        filter(!(Sample_geo_ID %in% c("GSM1701038", "GSM898305", "GSM997639")))

    # for plots
    plt_meta <- metadata #%>%
        # filter((Status == "ovarian tumour" & HistSubtypes == "high-grade serous carcinoma" | 
                # Status == "normal"))

    plt_expression <- expr_data %>% column_to_rownames("row_ID")
    plt_meta <- plt_meta[plt_meta$Sample_geo_ID %in% colnames(expr_data),]
    plt_expression <- expr_data[,plt_meta$Sample_geo_ID]

    # plots
    print("..plotting..")
    #  plot
    set.seed(1)  # Same feature subsample on every run.
    plot_res <- plots_normtumor(plt_expression[sample(1:nrow(plt_expression), 10000),], plt_meta, dataset, ', norm_vs_tumor')
    layout <- (plot_res[[1]] + plot_res[[2]] ) / 
            (plot_res[[3]] )
    ggsave(paste0("before/", dataset, "/intermediate/norm_vs_tumor_plot.png"), 
                plot = layout, width = 12, height = 12)

    if(is.null(all_metadata)){
        all_metadata <- metadata
        all_expression <- expr_data
    } else {
        
        all_metadata <- rbind(all_metadata, metadata)

        # shared features analysis:
        # all_expression <- inner_join(all_expression, expr_data, by = "row_ID")

        # all features analysis (union):
        all_expression <- full_join(all_expression, expr_data, by = "row_ID")
    }
}

In [ ]:
all_metadata %>%
    group_by(Dataset, Status) %>% summarise(n())

In [ ]:
all_metadata %>%
    group_by(Dataset, Status, HistSubtypes) %>% summarise(n())

In [ ]:
all_metadata <- all_metadata %>%
    mutate(Status = as.factor(Status),
             Stage2 = as.factor(Stage2),
             Grade2 = as.factor(Grade2),
             ClinicopathologicSubtypes = as.factor(ClinicopathologicSubtypes),
             Dataset = as.factor(Dataset))

all_expression <- all_expression %>% column_to_rownames("row_ID")
all_metadata <- all_metadata[all_metadata$Sample_geo_ID %in% colnames(all_expression),]
all_expression <- all_expression[,all_metadata$Sample_geo_ID]

print(paste0("Number of samples in type_1_early_vs_late: ", ncol(all_expression)))
print(paste0("Number of genes in type_1_early_vs_late: ", nrow(all_expression)))
stopifnot(nrow(all_metadata) == 331,
          sum(all_metadata$Status == "ovarian tumour") == 278,
          sum(all_metadata$Status == "normal") == 53,
          !any(all_metadata$Dataset == "GSE14407"),
          sum(all_metadata$Dataset == "GSE10971") == 23)


In [ ]:
#  plot
set.seed(1)  # Same feature subsample on every run.
plot_res <- plots_normtumor(all_expression[sample(1:nrow(all_expression), 10000),], 
                            all_metadata, "", 'norm_vs_tumor')
layout <- (plot_res[[1]] + plot_res[[2]] ) / 
         (plot_res[[3]] )
# ggsave("before/plots/norm_vs_HGSC_plot.png", 
#              plot = layout, width = 14, height = 12)
ggsave("before/plots/norm_vs_tumor_plot_UNION.png", 
             plot = layout, width = 14, height = 12)



# Prepare correction inputs

Export the inputs below, then run central correction from the repository root:

```bash
pixi run python -m evaluation_utils.run_correction_experiments correct --datasets ovarian_cancer
```


In [ ]:
# Encode the biological contrast in the exported client designs.
design <- model.matrix(~all_metadata$Status)
colnames(design) <- c("Intercept", "HGSC")


In [ ]:
# save expression and metadata in folder
# all in one file
write.table(all_metadata, "before/all_metadata.tsv", sep = "\t", quote = FALSE, row.names = FALSE)
# write.table(all_expression %>% rownames_to_column("Gene"), "before/all_expression.tsv", sep = "\t", quote = FALSE, row.names = FALSE)

# Central input, zipped because of its size. conda-forge R sets R_ZIPCMD to an
# empty string, so name the system zip command explicitly.
central_input <- file.path(tempdir(), "central_input.tsv")
write.table(all_expression %>% rownames_to_column("Gene"), central_input, sep = "\t", quote = FALSE, row.names = FALSE)
Sys.setFileTime(central_input, as.POSIXct("2000-01-01", tz = "UTC"))  # Reproducible ZIP bytes.
unlink("before/central_input.zip")
utils::zip("before/central_input.zip", central_input, flags = "-j9Xq", zip = "zip")
unlink(central_input)


In [ ]:
path_to_after_data <- "after"

design <- design %>% as.data.frame()
rownames(design) <- all_metadata$Sample_geo_ID

for(center in unique(all_metadata$Dataset)){
    print(center)
    path_to_before <- paste0("before/", center, "/")

    design_center = design[all_metadata[all_metadata$Dataset == center,]$Sample_geo_ID,]['HGSC']

    # if(!dir.exists(path_to_after)){
    #     dir.create(path_to_after)
    # }

    write.table(design_center %>% rownames_to_column('file'),
        file = paste0(path_to_before, "design.tsv"), sep = "\t", quote = F, row.names = F, col.names = T)

    # take also the expression data
    expr_center <- all_expression[,all_metadata$Sample_geo_ID[all_metadata$Dataset == center]]
    print(dim(expr_center))
    write.table(expr_center %>% rownames_to_column("Gene"), file = paste0(path_to_before, "expr_for_correction_UNION.tsv"), sep = "\t", quote = F, row.names = F, col.names = T)
}

# Session information


In [ ]:
sessionInfo()